# Discount Calendar Optimization

This notebook creates an optimized discount calendar to maximize revenue.


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from utils import load_data, save_data
from model_xgboost import train_xgboost, predict_xgboost
from discount_optimizer import create_discount_calendar, calculate_price_elasticity

# Load features
data_path = '../data/processed/grocery_features.csv'
df = load_data(data_path)

# Prepare features and target
target_col = 'sales'
feature_cols = [col for col in df.columns if col not in [target_col, 'date']]
X = df[feature_cols]
y = df[target_col]

# Train model for forecasting
xgb_model, _, _, _, _ = train_xgboost(X, y)


In [ ]:
# Calculate price elasticity
elasticity = calculate_price_elasticity(df, price_col='price', sales_col='sales', discount_col='discount')
print(f"Price Elasticity: {elasticity:.2f}")

# Create future dates for discount calendar
last_date = pd.to_datetime(df['date']).max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=90, freq='D')
future_df = pd.DataFrame({'date': future_dates})

# Create features for future dates (simplified - in practice, you'd need to create all features)
# This is a placeholder - you'll need to properly engineer features for future dates
print("\nNote: In practice, you need to create all temporal and other features for future dates")


In [ ]:
# Create discount calendar (example with sample data)
# In practice, you'd create proper features for future dates first
from feature_engineering import prepare_features

# For demonstration, using last 90 days of data
sample_df = df.tail(90).copy()
sample_df['date'] = future_dates[:len(sample_df)]

# Create features
sample_features = prepare_features(sample_df, target_col='sales', date_col='date')
sample_features = sample_features[feature_cols]

# Create discount calendar
base_price = 10.0  # Adjust based on your data
discount_calendar = create_discount_calendar(
    sample_df,
    xgb_model,
    feature_cols,
    date_col='date',
    base_price=base_price,
    elasticity=elasticity,
    min_discount=0,
    max_discount=50
)

print("\nDiscount Calendar Sample:")
print(discount_calendar[['date', 'predicted_sales', 'optimal_discount', 'predicted_revenue']].head(10))


In [ ]:
# Save discount calendar
output_path = '../data/processed/discount_calendar.csv'
save_data(discount_calendar, output_path)
print(f"\nDiscount calendar saved to {output_path}")
